## LEVEL 1
1. Merge collected H3 tables
2. Drop duplicates by place id
3. Drop rows with 0 ratings
4. Export places by H3 res 8 cell group

#### Initialize

In [1]:
import sys
from pathlib import Path
import pandas as pd
REPROCESS = True

HERE = Path.cwd()
PARENT = HERE.parent.parent.parent  # server/scripts
if str(PARENT) not in sys.path:
    sys.path.insert(0, str(PARENT))
CACHE_BUCKET_PATH = PARENT / "server/out/places_cache"
LEVEL1_BUCKET_PATH = PARENT / "server/out/places_level1"
LEVEL1_BUCKET_PATH.mkdir(parents=True, exist_ok=True)
ERROR_BUCKET = CACHE_BUCKET_PATH / "error"
ERROR_BUCKET.mkdir(parents=True, exist_ok=True)
CACHE_BUCKET = [f for f in CACHE_BUCKET_PATH.rglob("*.csv") if f.is_file()]

In [2]:
PROCESS_LEDGER = PARENT / "server/map/progress_ledger.csv"
ref_ledger = pd.read_csv(PROCESS_LEDGER, usecols=["tile_id", "tile_path_id", "seed_index", "places_count"])
ref_ledger["tile_id"] = ref_ledger["tile_id"].astype(str)
ref_ledger["tile_path_id"] = ref_ledger["tile_path_id"].astype(str)
ref_ledger = ref_ledger[ref_ledger["places_count"]>0]

#### Merge

In [3]:
CACHE_DFS = []
for f in CACHE_BUCKET:

    if f.stem.endswith("[error]"):
        # print(f"Error File Detected, moving to error bucket: {f.name}")
        f.replace(ERROR_BUCKET / f.name)
        continue
    try: df = pd.read_csv(f)
    except: df = pd.DataFrame()
    if df.empty: 
        continue

    ref_row = ref_ledger[ref_ledger["tile_path_id"] == f.stem]
    ref_info = ref_row.iloc[0] if not ref_row.empty else None
    if ref_info is None:
        raise ValueError(f"Missing Reference Info for file: {f.name}")
    
    df["tile_id"] = ref_info["tile_id"]
    df["tile_path_id"] = ref_info["tile_path_id"]
    df["seed_index"] = ref_info["seed_index"]
    path_ids = f.stem.split("-")
    df["level"] = len(path_ids) - 1

    CACHE_DFS.append(df)

cached_df = pd.concat(CACHE_DFS, ignore_index=True)
# Drop Duplicates
cached_df.drop_duplicates(subset=["id"], inplace=True)

ValueError: No objects to concatenate

Check Validity

In [ ]:
df_validity = pd.DataFrame({ 
    col: cached_df[col].notna().sum() / len(cached_df) 
    for col in cached_df.columns 
}, index=[0])
display(df_validity)

,id,displayName,primaryTypeDisplayName,rating,userRatingCount,location,shortFormattedAddress,googleMapsUri,priceRange,priceLevel,...,regularOpeningHours,pureServiceAreaBusiness,containingPlaces,accessibilityOptions,addressDescriptor,postalAddress,tile_id,tile_path_id,seed_index,level
0,1.0,1.0,1.0,0.90778,0.90778,1.0,1.0,1.0,0.773263,0.333449,...,0.835321,0.0,0.194633,0.648107,0.999653,0.994106,1.0,1.0,1.0,1.0


#### Drop

In [ ]:
# Drop Missing Ratings
df_level1 = cached_df[cached_df['rating'].notna() & cached_df['userRatingCount'].notna()]
print(f"""
    Dropped {len(cached_df) - len(df_level1)} Rows; 
    Dropped No Ratings Zones: {cached_df["tile_id"].nunique() - df_level1["tile_id"].nunique()}
""")

# Sort by ID and Reset Index
df_level1.sort_values(by="id", inplace=True)
df_level1.reset_index(drop=True, inplace=True)


    Dropped 1330 Rows; 
    Dropped No Ratings Zones: 63



#### Export

In [ ]:
for seed_id, group in df_level1.groupby("seed_index"):
    save_path = LEVEL1_BUCKET_PATH / f"{seed_id}.csv"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    group.to_csv(save_path, index=False)

In [ ]:
print(f"Processed {len(df_level1)}")

Processed 13092
